<u><b><h1 style="text-align:center; line-height:25px; color:#000000; background:#EFEFEF; border: 1px solid #FF6B6B ; padding:20px;">Sentiment analysis of customer reviews</h1></b></u>
**Course:** (DLBDSEAIS – Project: Artificial Intelligence  
**Tools**: Pandas, scikit-learn for the TF-IDF baseline, Hugging Face transformers (with PyTorch) for DistilBERT and Gradio for the interface  
**Dataset:** <a href="https://www.kaggle.com/datasets/fawadhossaini1415/amazon-fashion-800k-user-reviews-dataset">Amazon Fashion Products Reviews</a>  
**<a href="https://github.com/davidlupau/sentiment-analysis-customer-reviews">GitHub repository</a>**

<b><h2 style="padding: 10px; border-left: 3px solid #FF6B6B;">Setup & Imports</h2></b>

In [10]:
# Core
import pandas as pd
import torch
import platform

print("Import successful")

Import successful


### Select device
Select the compute backend automatically so the notebook runs unchanged on a cloud GPU (CUDA), an Apple Silicon Mac (MPS), or any CPU-only machine.

In [11]:
def detect_device() -> torch.device:
    """Detect the best available compute device.

    Checks hardware availability in order of preference:
      1. CUDA  – NVIDIA GPU (fastest for most deep-learning workloads).
      2. MPS   – Apple Silicon GPU (Metal Performance Shaders).
      3. CPU   – universal fallback.

    Returns:
        torch.device: The most capable device available on this machine.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = detect_device()
print(f"Device   : {DEVICE}")
print(f"Platform : {platform.platform()}")
print(f"PyTorch  : {torch.__version__}")

Device   : mps
Platform : macOS-26.5.1-arm64-arm-64bit-Mach-O
PyTorch  : 2.12.0


### Loading the dataset

In [12]:
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").is_dir()
)

df = pd.read_csv(PROJECT_ROOT / "data" / "amazon-fashion-reviews-dataset.csv")
print("Dataset loaded successfully")

Dataset loaded successfully


### Data cleaning

In [ ]:
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unusable rows from the dataset.

    Drops rows where 'text' or 'rating' contain null values, then removes
    exact duplicate rows. Reports counts at each step.

    Args:
        df: Raw DataFrame loaded from the CSV file.

    Returns:
        pd.DataFrame: Cleaned DataFrame with nulls and duplicates removed.
    """
    initial_rows = len(df)
    print(f"Initial row count: {initial_rows:,}")

    null_mask = df[["text", "rating"]].isnull().any(axis=1)
    null_count = int(null_mask.sum())
    if null_count:
        print(f"Dropping {null_count:,} rows with null values in 'text' or 'rating'.")
        df = df[~null_mask]
    else:
        print("No null values found in 'text' or 'rating'.")

    duplicate_count = int(df.duplicated().sum())
    if duplicate_count:
        print(f"Dropping {duplicate_count:,} duplicate rows.")
        df = df.drop_duplicates()
    else:
        print("No duplicate rows found.")

    cleaned_rows = len(df)
    print(f"Final row count: {cleaned_rows:,} ({initial_rows - cleaned_rows:,} rows removed).\n")

    return df.reset_index(drop=True)


df = clean_dataset(df)

---